In [68]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [69]:
%pip install numpy
%pip install pandas
%pip install pytest

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Solar Micro-Grid Dispatch Planner

In [70]:
from src.microgrid import MicroGrid

micro_grid = MicroGrid()
determinant = micro_grid.determinant()
condition = micro_grid.condition()

print(f"Determinant is: {determinant}", f"Condition: {condition}")

Determinant is: -4.999999999999999 Condition: 5.82842712474619


### Explain determinant and condition
**Determinant being -4.99999 or 5.** This means the co-efficients are meaningful and this usable to solve the system, if it was 0, it would be the system's equation are not solveable  
**Condtion being 5.8284...** A condition number of about 5.8284 means a 1% error in D1 or D2 can cause at most about a 5.8284% error in the values of x or y. That's small, so the system is well-conditioned. A large condition number , say more that 100, in 1000s or more, would cause significant measurement errors

### Interactive Input

In [71]:
from src.validate import Validate

try:
    validate = Validate()
    
    d1_interactive = input('Provide daytime load (KWh): ')
    validate.check_energy_input(d1_interactive)

    d2_interactive = input('Provide critical-equipment load (kWh):')
    validate.check_energy_input(d2_interactive)
    
    #solve
    result = micro_grid.solve(float(d1_interactive),float(d2_interactive))
    
    print("Results: ",result)
except Exception as e:
    print(e)    



Results:  [ 1.6 -0.4]


### Generate CSV file


In [72]:
from src.csv_generator import CsvGenerator
from pathlib import Path
import csv

csv_file_path = './files/solar_data.csv'
csv_generator = CsvGenerator()

#generate file
csv_generator.solar_grid_csv(csv_file_path,30,36)

#check if csv file exists
if Path(csv_file_path).exists():
    print(f"File {csv_file_path} generate successfully!")


File ./files/solar_data.csv generate successfully!


### Read csv file and solve for all 30 days

**Benchmarking, solve in Loop vs single vectorized cell**

In [ ]:
import pandas
import timeit
import numpy

def solve_with_loop(d_data):
    results = []
    #processing with loop
    for l_d1, l_d2 in d_data:
        results.append(micro_grid.solve(l_d1, l_d2))
    return results            

def solve_with_single_v_cell(d_data):
    #solve with single vectorized cell
    return numpy.linalg.solve(micro_grid.co_efficients, d_data.T).T
     

#only run bench marking n_runs number of times
n_runs = 5000

# check if file exists
if Path(csv_file_path).exists():
    with open(csv_file_path, "r") as file:
        rows = list(csv.DictReader(file))
        d_data = numpy.array([[float(r["d1"]), float(r["d2"])] for r in rows])
            
    #solve with loop
    time_taken_loop = timeit.timeit(lambda: solve_with_loop(d_data), number = n_runs)
    
    #solve with single vectorized cell
    time_taken_single_v_cell = timeit.timeit(lambda: solve_with_single_v_cell(d_data),  number = n_runs)
    
    #bench mark results
    benchmark = pandas.DataFrame({
                    "method": ["loop", "vectorized"],
                    "time (ms)": [time_taken_loop * 1e3, time_taken_single_v_cell * 1e3],
                    "speed-up": [1.0, time_taken_loop / time_taken_single_v_cell],
                })      
     
    display(benchmark.set_index("method").round(3).style.set_caption(f"Benchmark over {n_runs:,} runs ({len(d_data):,} days per run)")
    )


    # df = pandas.DataFrame(results,columns=["day", "d1", "d2", "x", "y"])
    # df

else:
     print(f"{csv_file_path} does not exist!")    

,time (ms),speed-up
method,,
loop,967.958,1.000
vectorized,29.704,32.587
